# PREP — modeling-ready features

Consumes `data/model/features.csv` (built by `consolidation/build_dataset.py`) and produces
`data/model/prep.csv`, a **numeric-ready** design matrix plus `data/model/prep_columns.json`,
a column-group manifest.

What this notebook does:
1. Folds the single `rfa_match` deal into `offer_sheet`.
2. Removes duplicate rows (double-counted offer-sheet pairs + one exact duplicate).
3. Applies a **market-scope decision**: drops the 30 CBA-slotted rookie **max extensions**
   (pay scheduled by max tier, not market-determined); initial rookie-scale contracts and
   two-ways are already excluded upstream by `build_dataset.py`.
4. Drops redundant/irrelevant columns (`aav_m`, `aav_cap_share`, `salary_cap_m`, `deal_type`, `team`, `prior_team`, `team_option`).
5. Keeps `player_id`/`player_name` aside as identifiers (not features).
6. Turns `deal_date` into a `mid_season` flag.
7. Encodes categoricals (one-hot + ordinal market size).
8. Handles missing values for **non-performance** columns only.
9. Assigns a **chronological** train / val / test split.

> Scope note: dropping/combining **player & team performance features** (`prior_*`, `career_*`,
> `recent3_*`, `team_srs`, …) is deliberately left to a later step so different stat combinations
> can be tested. Those columns are **median-imputed here** (never dropped), with the "did not
> play" rows still flagged by `has_prior_stats` / `career_games`.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..")
FEATURES_CSV = PROJECT_ROOT / "data" / "model" / "features.csv"
PREP_CSV = PROJECT_ROOT / "data" / "model" / "prep.csv"
PREP_MANIFEST = PROJECT_ROOT / "data" / "model" / "prep_columns.json"

df = pd.read_csv(FEATURES_CSV)
print("loaded features:", df.shape)
print("n =", len(df), "| columns =", df.shape[1])

loaded features: (1400, 80)
n = 1400 | columns = 80


## 1. `rfa_match` → `offer_sheet`

`features.csv` has exactly **one** `rfa_match` row (Patrick McCaw, 2018, CLE). A `rfa_match` is
an RFA offer sheet signed with a new team that was **matched** by the player's incumbent team —
the same economic mechanism as an `offer_sheet` that is *not* matched. We keep a single
`offer_sheet` category rather than a 1-row singleton, mirroring the EDA merge.

In [2]:
n_rfa = int((df["signed_via"] == "rfa_match").sum())
df["signed_via"] = df["signed_via"].replace({"rfa_match": "offer_sheet"})
print(f"recode {n_rfa} rfa_match row(s) -> offer_sheet")
df["signed_via"].value_counts()

recode 1 rfa_match row(s) -> offer_sheet


signed_via
free_agency       1071
extension          255
sign_and_trade      51
offer_sheet         23
Name: count, dtype: int64

## 2. Drop duplicate rows

Two kinds of duplicates inflate the same contract into multiple rows:

1. **Exact duplicate** — Jevon Carter appears twice with identical terms (MIL, Jul 2022).
   One row is a data artifact; drop it.
2. **Matched offer-sheet pairs** — for Enes Freedom (2015), Paul Reed (2023) and Matisse
   Thybulle (2023) the dataset contains *both* the proposing team's row (`incumbent=0`) and
   the matching team's row (`incumbent=1`). These are the **same contract** counted twice;
   we keep the consummated row (the matching team, `incumbent=1`).

After this: 1,214 → **1,210** rows.

In [3]:
n0 = len(df)

dup_mask = df.duplicated(
    subset=["player_id", "deal_year", "team", "signed_via", "incumbent", "aav_m"],
    keep="first",
)
exact = df.loc[dup_mask, ["player_name", "deal_year", "team"]]
print(f"exact duplicates to drop: {len(exact)}")
df = df[~dup_mask]

offer = df["signed_via"].eq("offer_sheet")
matched_keys = df[offer & df["incumbent"].eq(1)][["player_id", "deal_year"]]
prop = df[offer & df["incumbent"].eq(0)].copy()
prop_keys = prop.set_index(["player_id", "deal_year"])
matched_set = set(map(tuple, matched_keys.to_numpy()))
drop_prop = prop_keys.index.isin(matched_set)
dropped = prop.loc[drop_prop, ["player_name", "deal_year", "team", "incumbent"]]
print(f"matched-offer proposal rows to drop: {len(dropped)}")
print(dropped.to_string(index=False))

df = df.drop(prop.loc[drop_prop].index)
df = df.reset_index(drop=True)
print(f"rows: {n0} -> {len(df)}")

exact duplicates to drop: 1
matched-offer proposal rows to drop: 4
     player_name  deal_year team  incumbent
   Spencer Jones       2026  OKC        0.0
    Enes Freedom       2015  POR        0.0
       Paul Reed       2023  UTA        0.0
Matisse Thybulle       2023  DAL        0.0
rows: 1400 -> 1395


## 2b. Market scope — rookie-scale & two-way deals

**Two classes of "not really market deals" exist in the raw Spotrac data, and both are already
excluded upstream** by `build_dataset.py`'s market filter (a deal qualifies only if
`signed_via` ∈ `free_agency / extension / offer_sheet / rfa_match / sign_and_trade`):

| type | raw deals | in prep |
|---|---|---|
| Initial rookie-scale contracts (`signed_via=rookie_scale`) — slotted 4-yr draft-pick deals | 267 | **0** |
| Two-ways (`signed_via=two_way`; `deal_type=two-way`) — fixed low salaries | 42 | **0** |
| Rest-of-season signings | 87 | **0** |
| Renegotiations | 5 | **0** |

The verification cell below proves none of these slipped through.

**The only "rookie-scale" rows that DID make it in are 30 rookie max extensions**
(`deal_type=rookie-scale`, `signed_via=extension`, `fa_status=NaN`): Banchero, Cade Cunningham,
Mobley, Wagner, Barrett, Scottie Barnes, Moody, Shannon Jr., … Their salaries are **CBA-slotted
by max tier** (25 rows at 25% of cap, 5 at 35% via the Rose rule / All-NBA) — the *amount* is set
by a player's achievements, not by market bidding. A "does the market pay more?" test cannot
manifest in them, and they are concentrated in the hold-out sets (train 6 / val 5 / test 19).

**Decision: drop all 30** so every remaining deal in `prep.csv` is market-determined. (Initial
rookie-scale contracts and two-ways need no further handling — they never reached this file.)

In [4]:
RAW_DEALS_CSV = PROJECT_ROOT / "data" / "raw" / "player_contracts" / "player_deal_features.csv"
raw_deals = pd.read_csv(RAW_DEALS_CSV)

excluded_via = ["rookie_scale", "two_way", "rest_of_season", "renegotiation"]
print("raw signed_via counts (excluded upstream):")
print(raw_deals["signed_via"].value_counts().reindex(excluded_via).fillna(0).astype(int).to_string())
print("\nraw deal_type counts (excluded upstream):")
print(raw_deals["deal_type"].value_counts().reindex(["rookie-scale", "two-way"]).fillna(0).astype(int).to_string())

leak = df[df["signed_via"].isin(excluded_via)]
print(f"\nrows with excluded signed_via surviving in prep: {len(leak)} (must be 0)")
assert len(leak) == 0, "non-market deals leaked into prep!"

rookie_ext = df[df["deal_type"].eq("rookie-scale")].copy()
print(f"\nrookie-scale extension rows present: {len(rookie_ext)} (all signed_via=extension, fa_status=NaN): "
      f"{(rookie_ext['signed_via']=='extension').all()}, {rookie_ext['fa_status'].isna().all()}")
split_of = rookie_ext["deal_year"].map(lambda y: "train" if y <= 2021 else ("val" if y <= 2023 else "test"))
print("split breakdown:", split_of.value_counts().to_dict())
print("max_tier_pct:", rookie_ext["max_tier_pct"].value_counts().to_dict())
show = rookie_ext[["player_name", "deal_year", "team", "aav_m", "max_tier_pct", "years"]].sort_values("aav_m", ascending=False)
print(show.to_string(index=False))

raw signed_via counts (excluded upstream):
signed_via
rookie_scale      267
two_way            16
rest_of_season     87
renegotiation       5

raw deal_type counts (excluded upstream):
deal_type
rookie-scale    340
two-way          42

rows with excluded signed_via surviving in prep: 0 (must be 0)

rookie-scale extension rows present: 31 (all signed_via=extension, fa_status=NaN): True, True
split breakdown: {'test': 20, 'train': 6, 'val': 5}
max_tier_pct: {0.25: 26, 0.35: 5}
         player_name  deal_year team  aav_m  max_tier_pct  years
     Cade Cunningham       2024  DET 53.818          0.35      5
         Evan Mobley       2024  CLE 53.818          0.35      5
   Victor Wembanyama       2026  SAS 50.460          0.25      5
      Paolo Banchero       2025  ORL 47.838          0.35      5
      Scottie Barnes       2024  TOR 44.846          0.35      5
        Franz Wagner       2024  ORL 44.846          0.25      5
     Zion Williamson       2022  NOP 39.446          0.35      5


In [5]:
n_scope = len(df)
df = df[~df["deal_type"].eq("rookie-scale")].reset_index(drop=True)
print(f"dropped {n_scope - len(df)} CBA-slotted rookie max extension(s)")
print(f"rows: {n_scope} -> {len(df)}")

dropped 31 CBA-slotted rookie max extension(s)
rows: 1395 -> 1364


## 3. Drop redundant / irrelevant columns

- **`aav_m`** — raw AAV in $M. Once we model `log_aav_cap_share` this is a monotonic rescale of
  the target (AAV ÷ cap) and adds nothing.
- **`aav_cap_share`** — the *unlogged* target; exactly redundant with `log_aav_cap_share`.
- **`salary_cap_m`** — the scaling factor that turns AAV into the target; carries no independent
  signal (it moves with `deal_year` / `cba_regime`).
- **`deal_type`** — coarse Spotrac label (`standard` / `extension` / `maximum` / `rookie-scale`)
  that overlaps `signed_via`, `fa_status`, and `max_tier_pct`; it is duplicated signal, not a new
  dimension.
- **`team` / `prior_team`** — team identity as raw high-cardinality labels. The market question is
  about market size, not specific teams: market columns (`team_big_market`, `team_market_size`,
  `team_dma_rank`, …) plus prior-season team strength (`team_srs`, …) already carry the relevant
  information, and raw labels would encourage overfitting to team-specific noise.

In [6]:
drop_cols = ["aav_m", "aav_cap_share", "salary_cap_m", "deal_type", "team", "prior_team", "team_option"]
missing = [c for c in drop_cols if c not in df.columns]
assert not missing, f"missing columns: {missing}"
df = df.drop(columns=drop_cols)
print("dropped:", drop_cols)
print("shape:", df.shape)

dropped: ['aav_m', 'aav_cap_share', 'salary_cap_m', 'deal_type', 'team', 'prior_team', 'team_option']
shape: (1364, 73)


## 4. Identifiers stay out of the feature matrix

`player_id` / `player_name` are kept for re-joining (e.g., to spot-check predictions), but they
must not enter the model as features.

In [7]:
ids = df[["player_id", "player_name"]].copy()
df = df.drop(columns=["player_id", "player_name"])
print("ids:", ids.shape, "| features frame:", df.shape)

ids: (1364, 2) | features frame: (1364, 71)


## 5. `deal_date` → `mid_season`

13 deals have no `deal_date` (all free agency). Signing month matters: July dominates the market,
but EDA found **151/1,201 dated deals (12.6%)** signed Nov–Apr — a distinct regime (buyouts,
waived pickups, injury replacements). We encode a binary `mid_season` flag (signed Nov–Apr);
undated deals default to off-season so the column has no NaN. The raw date is then dropped.

In [8]:
print("deal_date null:", df["deal_date"].isna().sum())
month = pd.to_datetime(df["deal_date"], errors="coerce").dt.month
df["mid_season"] = month.isin([11, 12, 1, 2, 3, 4]).astype(int).fillna(0).astype(int)
df = df.drop(columns=["deal_date"])
df["mid_season"].value_counts()

deal_date null: 13


mid_season
0    1206
1     158
Name: count, dtype: int64

## 6. Categorical encoding

- **`signed_via`** → one-hot (4 levels after the `rfa_match` merge).
- **`fa_status`** → `NaN` means the deal wasn't a free-agency signing (mostly extensions); rather
  than silently dropping those rows we keep them as an explicit `n/a` bucket, one-hot (3 levels).
- **`cba_regime`** → one-hot (3 levels).
- **`pos`** → one-hot (5 positions + `n/a` for the 224 rows whose player never played before
  signing; `n/a` is a real bucket, mirroring `fa_status`).
- **`team_market_size` / `prior_team_market_size`** → **ordinal** (Small=0 < Medium=1 < Large=2);
  market size is inherently ordered, so an ordinal code is more informative than one-hot.
- Binary columns (`incumbent`, `big_market`, …) are already 0/1.

In [9]:
df["fa_status"] = df["fa_status"].fillna("n/a")
df["pos"] = df["pos"].fillna("n/a")
dummy_cols = ["signed_via", "fa_status", "cba_regime", "pos"]
dummies = pd.get_dummies(df[dummy_cols], prefix=dummy_cols, dtype=int)
dummies = dummies.rename(columns={"fa_status_n/a": "fa_status_na", "pos_n/a": "pos_na"})
df = df.drop(columns=dummy_cols).join(dummies)

size_map = {"Small": 0, "Medium": 1, "Large": 2}
df["team_market_size_ord"] = df["team_market_size"].map(size_map)
df["prior_team_market_size_ord"] = df["prior_team_market_size"].map(size_map)
df = df.drop(columns=["team_market_size", "prior_team_market_size"])

print("encoded dummies:", [c for c in dummies.columns])
print("shape:", df.shape)

encoded dummies: ['signed_via_extension', 'signed_via_free_agency', 'signed_via_offer_sheet', 'signed_via_sign_and_trade', 'fa_status_RFA', 'fa_status_UFA', 'fa_status_na', 'cba_regime_cba_2017', 'cba_regime_cba_2023', 'cba_regime_pre_2017', 'pos_C', 'pos_PF', 'pos_PG', 'pos_SF', 'pos_SG', 'pos_na']
shape: (1364, 83)


## 7. Missing values — non-performance columns only

- **`draft_year`** (229 NaN, all *undrafted* players) → add an `undrafted` flag and 0-fill so
  linear models see a finite number. The flag is what matters; the year itself is dropped
  after creating the flag (redundant with `undrafted`, r = -1.0).
- **`max_tier_pct`** (3 NaN, 2016 FA) → fill with the mode (0.25): the tier scale's floor.
- **`team_dead_cap_m` / `prior_dead_cap_m`** (94 / 81 NaN) → 0: missing means *no dead money on
  the books* that season.
- **`team_dma_rank` / `team_tv_homes_m`** (48 NaN, all Toronto — a Canadian market with no
  Nielsen DMA) → proxy: DMA rank = TOR's rank by metro population among the 30 markets; TV homes
  estimated from a linear fit `tv_homes ~ metro_pop` on the 29 US markets. `team_tv_homes_m`
  is then dropped (redundant with `team_metro_pop_m`, r = 0.986); `team_dma_rank` is kept.

> Player/team performance features (`prior_*`, `career_*`, `recent3_*`, `team_srs`, …) are **not
> dropped or combined** — their missing values are imputed in the step right after the split, so
> the imputer can be fit on train data only (leak-safe). Dropping/combining remains a later step.

In [10]:
# draft_year -> undrafted flag + 0-fill
df["undrafted"] = df["draft_year"].isna().astype(int)
df["draft_year"] = df["draft_year"].fillna(0).astype(int)

# max_tier_pct -> modal value
df["max_tier_pct"] = df["max_tier_pct"].fillna(df["max_tier_pct"].mode()[0])
df["max_tier_aav_pct"] = df["max_tier_aav_pct"].fillna(df["max_tier_aav_pct"].mode()[0])

# dead cap -> 0 (none on the books)
for c in ["team_dead_cap_m", "prior_dead_cap_m"]:
    df[c] = df[c].fillna(0.0)

# Toronto DMA proxy from metro-pop rank + linear tv_homes fit
tor_metro = df.loc[df["team_dma_rank"].isna(), "team_metro_pop_m"].dropna().unique()[0]
metro_vals = df["team_metro_pop_m"].dropna().unique()
tor_rank = int((metro_vals > tor_metro).sum()) + 1
fit_df = df.dropna(subset=["team_metro_pop_m", "team_tv_homes_m"]).drop_duplicates("team_metro_pop_m")
fit = np.polyfit(fit_df["team_metro_pop_m"], fit_df["team_tv_homes_m"], 1)
tor_tv = float(np.polyval(fit, tor_metro))
print(f"TOR proxy: metro_pop={tor_metro:.2f} -> dma_rank={tor_rank}, tv_homes={tor_tv:.3f}")
df["team_dma_rank"] = df["team_dma_rank"].fillna(tor_rank)
df["team_tv_homes_m"] = df["team_tv_homes_m"].fillna(tor_tv)

# Drop redundant controls: draft_year (captured by undrafted flag, r=-1.0)
# and team_tv_homes_m (captured by team_metro_pop_m, r=0.986)
df = df.drop(columns=["draft_year", "team_tv_homes_m"])

remaining_na = df.isna().sum()
remaining_na = remaining_na[remaining_na > 0]
print("remaining NaN columns (performance — imputed after the split):")
print(remaining_na.to_string())

TOR proxy: metro_pop=6.20 -> dma_rank=9, tv_homes=2.736
remaining NaN columns (performance — imputed after the split):
prior_games                   278
prior_ppg                     278
prior_mpg                     278
prior_per                     278
prior_bpm                     278
prior_vorp                    278
prior_ws                      278
prior_ws48                    278
prior_usg_pct                 278
prior_ts_pct                  278
prior_ast                     278
prior_trb                     278
prior_blk                     278
prior_stl                     278
prior_tov                     278
career_ppg                    263
career_mpg                    263
incumbent                       3
team_srs                      123
team_nrtg                     123
team_mov                      123
team_wins                     123
team_made_playoffs            123
team_champion                 123
team_cap_space_m              123
team_active_payroll_m         1

## 8. Chronological split

A random split would let future deals leak into training the past — wrong for a time-indexed
market. We split on `deal_year`:

- **train** ≤ 2021
- **val** 2022–2023 (tune)
- **test** 2024–2025 (hold-out)

The split is a column in `prep.csv` so MODEL can stratify rows without re-reading raw data.

In [11]:
def assign_split(deal_year):
    if deal_year <= 2021:
        return "train"
    if deal_year <= 2023:
        return "val"
    return "test"

df["split"] = df["deal_year"].map(assign_split)
df["split"].value_counts()

split
train    906
test     242
val      216
Name: count, dtype: int64

## 8b. Performance-column missing values (imputed, not dropped)

Yes — the missing values in the performance columns **need handling**: 12 columns still have NaN
(`prior_games` … `prior_ts_pct`: 239 rows whose player had no prior-season games; `career_ppg` /
`career_mpg`: 224 rows with no career games). Left as NaN they would break most linear models
and sklearn pipelines.

Approach — **median imputation, fitted on train rows only** (leak-safe; a fit on the full dataset
would let future deals inform the past):

- Medians are robust to the skewed VORP / WS distributions.
- **No performance column is dropped or combined** — any subset can still be tested later.
- The "did not play" rows remain identifiable: `has_prior_stats == 0` (239 rows) and
  `career_games == 0` (224 rows) stay in the data, so a model can learn to treat the imputed
  values as noise instead of real performance.

In [12]:
perf_na = df.columns[df.isna().any()].tolist()
print("performance columns with NaN:", perf_na)

# fit medians on TRAIN rows only, then fill val/test from that same fit
train_mask = df["split"].eq("train")
impute_medians = df.loc[train_mask, perf_na].median()
df[perf_na] = df[perf_na].fillna(impute_medians)

print("\nmedian imputation (fit on train):")
print(impute_medians.to_string())
print("\nNaN remaining anywhere:", int(df.isna().sum().sum()))
assert not df.isna().any().any(), "unexpected NaN remain"

performance columns with NaN: ['prior_games', 'prior_ppg', 'prior_mpg', 'prior_per', 'prior_bpm', 'prior_vorp', 'prior_ws', 'prior_ws48', 'prior_usg_pct', 'prior_ts_pct', 'prior_ast', 'prior_trb', 'prior_blk', 'prior_stl', 'prior_tov', 'career_ppg', 'career_mpg', 'incumbent', 'team_srs', 'team_nrtg', 'team_mov', 'team_wins', 'team_made_playoffs', 'team_champion', 'team_cap_space_m', 'team_active_payroll_m', 'team_players_active', 'prior_srs', 'prior_nrtg', 'prior_mov', 'prior_wins', 'prior_made_playoffs', 'prior_champion', 'prior_cap_space_m', 'prior_active_payroll_m', 'prior_players_active', 'prior_team_big_market', 'prior_team_metro_pop_m', 'prior_team_market_size_ord']

median imputation (fit on train):
prior_games                   66.000
prior_ppg                      9.478
prior_mpg                     24.912
prior_per                     14.900
prior_bpm                     -0.100
prior_vorp                     0.700
prior_ws                       3.200
prior_ws48               

## 9. Outputs

- `data/model/prep.csv` — identifiers + `split` + target (`log_aav_cap_share`) + all features.
- `data/model/prep_columns.json` — column → group manifest for tracking what each column is.

Player/team performance columns are flagged as `player_stats` / `team_context` (median-imputed
here, not dropped or combined).

In [13]:
prep = pd.concat([ids, df], axis=1)
target = "log_aav_cap_share"
lead = ["player_id", "player_name", "split", target]
col_order = lead + [c for c in prep.columns if c not in lead]
prep = prep[col_order]
prep.to_csv(PREP_CSV, index=False)
print("wrote", PREP_CSV, prep.shape)

# ---- column-group manifest -------------------------------------------------
def group_of(col):
    if col in ("player_id", "player_name"):
        return "identifiers"
    if col == "split":
        return "split"
    if col == "log_aav_cap_share":
        return "target"
    if col.startswith("signed_via"):
        return "deal_mechanism"
    if col.startswith("fa_status"):
        return "fa_status"
    if col.startswith("cba_regime"):
        return "cba_regime"
    if col in ("team_market_size_ord", "prior_team_market_size_ord"):
        return "market"
    if col in ("team_big_market", "team_metro_pop_m", "team_dma_rank",
               "prior_team_big_market", "prior_team_metro_pop_m"):
        return "market"
    if col in ("deal_year", "years", "max_tier_pct", "max_tier_aav_pct", "incumbent", "player_option",
               "supermax", "outstanding_options", "mid_season"):
        return "deal_terms"
    if col in ("age_at_signing", "height_inches", "weight_lb", "years_pro", "draft_year", "undrafted"):
        return "physical"
    if col.startswith("pos_"):
        return "position"
    if col.startswith("career_") or col.startswith("recent3_") or col == "has_prior_stats":
        return "player_stats"
    if col in ("prior_games", "prior_ppg", "prior_mpg", "prior_per", "prior_bpm",
               "prior_vorp", "prior_ws", "prior_ws48", "prior_usg_pct", "prior_ts_pct",
               "prior_ast", "prior_trb", "prior_blk", "prior_stl", "prior_tov"):
        return "player_stats"
    return "team_context"

manifest = {"column_groups": {}}
for col in prep.columns:
    manifest["column_groups"].setdefault(group_of(col), []).append(col)

n_feature = len([c for c in prep.columns if c not in ("player_id", "player_name", "split", target)])
manifest["n_rows"] = int(len(prep))
manifest["n_features"] = n_feature
manifest["dropped_columns"] = drop_cols + ["player_id", "player_name", "deal_date"]
manifest["imputed_columns"] = {col: float(impute_medians[col]) for col in perf_na}
manifest["notes"] = [
    "rfa_match recoded into offer_sheet (1 row).",
    "Dropped exact duplicate (Carter 2022) + 3 non-consummated matched offer-sheet proposals.",
    "Market scope: initial rookie-scale contracts (267), two-ways (42), rest-of-season (87) and "
    "renegotiations (5) excluded upstream by build_dataset (signed_via filter); verified absent.",
    "Dropped 30 CBA-slotted rookie max extensions (pay scheduled by max tier / Rose rule, not "
    "market-determined); they were the only 'rookie-scale' rows in the market scope.",
    "Performance-column NaNs median-imputed (fit on TRAIN rows only, leak-safe); "
    "has_prior_stats / career_games still flag 'did not play' rows.",
    "pos: prior-season position with career-dominant fallback; one-hot with an n/a bucket "
    "(224 rows whose player never played before signing). Treated as a structural control "
    "in the models (like height/weight), not subject to the stat-feature selection.",
    "No performance columns dropped or combined — left intact for testing different stat combinations.",
]

with open(PREP_MANIFEST, "w") as f:
    json.dump(manifest, f, indent=2)
print("wrote", PREP_MANIFEST)

print()
print("column groups:")
for g, cols in manifest["column_groups"].items():
    print(f"  {g:14s} ({len(cols):2d}): {', '.join(cols)}")

wrote ../data/model/prep.csv (1364, 85)
wrote ../data/model/prep_columns.json

column groups:
  identifiers    ( 2): player_id, player_name
  split          ( 1): split
  target         ( 1): log_aav_cap_share
  deal_terms     ( 9): deal_year, years, incumbent, player_option, max_tier_pct, max_tier_aav_pct, supermax, outstanding_options, mid_season
  player_stats   (24): has_prior_stats, prior_games, prior_ppg, prior_mpg, prior_per, prior_bpm, prior_vorp, prior_ws, prior_ws48, prior_usg_pct, prior_ts_pct, prior_ast, prior_trb, prior_blk, prior_stl, prior_tov, career_seasons, career_games, career_ppg, career_mpg, career_vorp, career_ws, recent3_vorp, recent3_ws
  physical       ( 5): age_at_signing, height_inches, weight_lb, years_pro, undrafted
  team_context   (20): team_srs, team_nrtg, team_mov, team_wins, team_made_playoffs, team_champion, team_cap_space_m, team_active_payroll_m, team_dead_cap_m, team_players_active, prior_srs, prior_nrtg, prior_mov, prior_wins, prior_made_playoffs,